In [ ]:
import os
import pandas as pd

# Use double quotes around the raw string – no escape issues
folder_path = r"C:\Users\B1Bash\Desktop\Gorgeous\my_portfolio\Portfolio_2"
file_name = 'heart_failure_clinical_records_dataset.csv'
full_path = os.path.join(folder_path, file_name)

# Check if the file exists
if os.path.exists(full_path):
    df = pd.read_csv(full_path)
    print("✅ Dataset loaded successfully!")
    print(f"Shape: {df.shape}")
    print("First 5 rows:")
    print(df.head())
else:
    print(f"❌ File not found at: {full_path}")
    print("Please verify the file name and path.")
    # List contents of the folder to help debug
    if os.path.exists(folder_path):
        print("\nContents of the folder:")
        for item in os.listdir(folder_path):
            print(f"  - {item}")
    else:
        print(f"The folder '{folder_path}' does not exist. Check your path.")

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, roc_auc_score

# Models
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

# Set style for better plots
sns.set_style("whitegrid")
%matplotlib inline

In [ ]:
print("Dataset Info:")
df.info()

print("\nDescriptive Statistics:")
df.describe()

In [ ]:
print("Missing Values per Column:")
print(df.isnull().sum())

In [ ]:
plt.figure(figsize=(8, 6))
sns.countplot(x='DEATH_EVENT', data=df)
plt.title('Distribution of Death Event (0 = Alive, 1 = Dead)', fontsize=14)
plt.xlabel('Death Event')
plt.ylabel('Count')
plt.show()

print("Target Distribution:")
print(df['DEATH_EVENT'].value_counts())
print(f"Mortality Rate: {df['DEATH_EVENT'].mean()*100:.2f}%")

In [ ]:
plt.figure(figsize=(14, 10))
sns.heatmap(df.corr(), annot=True, cmap='coolwarm', fmt='.2f', linewidths=0.5)
plt.title('Feature Correlation Heatmap', fontsize=16)
plt.tight_layout()
plt.show()

In [ ]:
sns.pairplot(df[['age', 'ejection_fraction', 'serum_creatinine', 'serum_sodium', 'DEATH_EVENT']], 
             hue='DEATH_EVENT', diag_kind='kde')
plt.suptitle('Pairplot of Key Features', y=1.02, fontsize=14)
plt.show()

In [ ]:
plt.figure(figsize=(8, 6))
sns.boxplot(x='DEATH_EVENT', y='age', data=df)
plt.title('Boxplot of Age by Death Event', fontsize=14)
plt.xlabel('Death Event (0=Alive, 1=Dead)')
plt.ylabel('Age')
plt.show()

In [ ]:
plt.figure(figsize=(8, 6))
sns.boxplot(x='DEATH_EVENT', y='creatinine_phosphokinase', data=df)
plt.title('Boxplot of Creatinine Phosphokinase by Death Event', fontsize=14)
plt.xlabel('Death Event (0=Alive, 1=Dead)')
plt.ylabel('Creatinine Phosphokinase')
plt.show()

In [ ]:
plt.figure(figsize=(8, 6))
sns.boxplot(x='DEATH_EVENT', y='ejection_fraction', data=df)
plt.title('Boxplot of Ejection Fraction by Death Event', fontsize=14)
plt.xlabel('Death Event (0=Alive, 1=Dead)')
plt.ylabel('Ejection Fraction')
plt.show()

In [ ]:
plt.figure(figsize=(8, 6))
sns.boxplot(x='DEATH_EVENT', y='platelets', data=df)
plt.title('Boxplot of Platelets by Death Event', fontsize=14)
plt.xlabel('Death Event (0=Alive, 1=Dead)')
plt.ylabel('Platelets')
plt.show()

In [ ]:
plt.figure(figsize=(8, 6))
sns.boxplot(x='DEATH_EVENT', y='serum_creatinine', data=df)
plt.title('Boxplot of Serum Creatinine by Death Event', fontsize=14)
plt.xlabel('Death Event (0=Alive, 1=Dead)')
plt.ylabel('Serum Creatinine')
plt.show()

In [ ]:
plt.figure(figsize=(8, 6))
sns.boxplot(x='DEATH_EVENT', y='serum_sodium', data=df)
plt.title('Boxplot of Serum Sodium by Death Event', fontsize=14)
plt.xlabel('Death Event (0=Alive, 1=Dead)')
plt.ylabel('Serum Sodium')
plt.show()

In [ ]:
X = df.drop('DEATH_EVENT', axis=1)
y = df['DEATH_EVENT']

print(f"Features shape: {X.shape}")
print(f"Target shape: {y.shape}")
print(f"Features columns: {list(X.columns)}")

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Training set size: {X_train.shape[0]} samples")
print(f"Test set size: {X_test.shape[0]} samples")
print("\nTraining target distribution:")
print(y_train.value_counts())
print(f"Training mortality rate: {y_train.mean()*100:.2f}%")
print("\nTest target distribution:")
print(y_test.value_counts())
print(f"Test mortality rate: {y_test.mean()*100:.2f}%")

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Feature scaling completed!")
print(f"Scaled training data shape: {X_train_scaled.shape}")
print(f"Scaled test data shape: {X_test_scaled.shape}")
print(f"\nMean of scaled training data (first 5 features): {X_train_scaled.mean(axis=0)[:5].round(4)}")
print(f"Std of scaled training data (first 5 features): {X_train_scaled.std(axis=0)[:5].round(4)}")

In [ ]:
models = {
    'SVM': SVC(kernel='rbf', probability=True, random_state=42),
    'KNN': KNeighborsClassifier(n_neighbors=5),
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'Random Forest': RandomForestClassifier(random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(random_state=42)
}

print("Models defined:")
for name in models.keys():
    print(f"  - {name}")

In [ ]:
name = 'SVM'
model = models[name]

cv_scores = cross_val_score(model, X_train_scaled, y_train, cv=5, scoring='accuracy')
model.fit(X_train_scaled, y_train)
y_pred = model.predict(X_test_scaled)
y_pred_proba = model.predict_proba(X_test_scaled)[:, 1]

accuracy = accuracy_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_pred_proba)

print(f"\n{'='*50}")
print(f"{name} Performance:")
print(f"{'='*50}")
print(f"  CV Accuracy: {np.mean(cv_scores):.4f} (±{np.std(cv_scores):.4f})")
print(f"  Test Accuracy: {accuracy:.4f}")
print(f"  AUC-ROC: {auc:.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

In [ ]:
name = 'KNN'
model = models[name]

cv_scores = cross_val_score(model, X_train_scaled, y_train, cv=5, scoring='accuracy')
model.fit(X_train_scaled, y_train)
y_pred = model.predict(X_test_scaled)
y_pred_proba = model.predict_proba(X_test_scaled)[:, 1]

accuracy = accuracy_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_pred_proba)

print(f"\n{'='*50}")
print(f"{name} Performance:")
print(f"{'='*50}")
print(f"  CV Accuracy: {np.mean(cv_scores):.4f} (±{np.std(cv_scores):.4f})")
print(f"  Test Accuracy: {accuracy:.4f}")
print(f"  AUC-ROC: {auc:.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

In [ ]:
name = 'Logistic Regression'
model = models[name]

cv_scores = cross_val_score(model, X_train_scaled, y_train, cv=5, scoring='accuracy')
model.fit(X_train_scaled, y_train)
y_pred = model.predict(X_test_scaled)
y_pred_proba = model.predict_proba(X_test_scaled)[:, 1]

accuracy = accuracy_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_pred_proba)

print(f"\n{'='*50}")
print(f"{name} Performance:")
print(f"{'='*50}")
print(f"  CV Accuracy: {np.mean(cv_scores):.4f} (±{np.std(cv_scores):.4f})")
print(f"  Test Accuracy: {accuracy:.4f}")
print(f"  AUC-ROC: {auc:.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

In [ ]:
name = 'Decision Tree'
model = models[name]

cv_scores = cross_val_score(model, X_train_scaled, y_train, cv=5, scoring='accuracy')
model.fit(X_train_scaled, y_train)
y_pred = model.predict(X_test_scaled)
y_pred_proba = model.predict_proba(X_test_scaled)[:, 1]

accuracy = accuracy_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_pred_proba)

print(f"\n{'='*50}")
print(f"{name} Performance:")
print(f"{'='*50}")
print(f"  CV Accuracy: {np.mean(cv_scores):.4f} (±{np.std(cv_scores):.4f})")
print(f"  Test Accuracy: {accuracy:.4f}")
print(f"  AUC-ROC: {auc:.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

In [ ]:
name = 'Random Forest'
model = models[name]

cv_scores = cross_val_score(model, X_train_scaled, y_train, cv=5, scoring='accuracy')
model.fit(X_train_scaled, y_train)
y_pred = model.predict(X_test_scaled)
y_pred_proba = model.predict_proba(X_test_scaled)[:, 1]

accuracy = accuracy_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_pred_proba)

print(f"\n{'='*50}")
print(f"{name} Performance:")
print(f"{'='*50}")
print(f"  CV Accuracy: {np.mean(cv_scores):.4f} (±{np.std(cv_scores):.4f})")
print(f"  Test Accuracy: {accuracy:.4f}")
print(f"  AUC-ROC: {auc:.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

In [ ]:
name = 'Gradient Boosting'
model = models[name]

cv_scores = cross_val_score(model, X_train_scaled, y_train, cv=5, scoring='accuracy')
model.fit(X_train_scaled, y_train)
y_pred = model.predict(X_test_scaled)
y_pred_proba = model.predict_proba(X_test_scaled)[:, 1]

accuracy = accuracy_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_pred_proba)

print(f"\n{'='*50}")
print(f"{name} Performance:")
print(f"{'='*50}")
print(f"  CV Accuracy: {np.mean(cv_scores):.4f} (±{np.std(cv_scores):.4f})")
print(f"  Test Accuracy: {accuracy:.4f}")
print(f"  AUC-ROC: {auc:.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

In [ ]:
results = {}

for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)
    y_pred_proba = model.predict_proba(X_test_scaled)[:, 1] if hasattr(model, 'predict_proba') else None
    
    cv_scores = cross_val_score(model, X_train_scaled, y_train, cv=5, scoring='accuracy')
    
    results[name] = {
        'cv_mean_accuracy': np.mean(cv_scores),
        'cv_std': np.std(cv_scores),
        'test_accuracy': accuracy_score(y_test, y_pred),
        'auc_roc': roc_auc_score(y_test, y_pred_proba) if y_pred_proba is not None else None
    }

comparison_df = pd.DataFrame({
    'Model': list(results.keys()),
    'CV Accuracy': [results[m]['cv_mean_accuracy'] for m in results],
    'CV Std': [results[m]['cv_std'] for m in results],
    'Test Accuracy': [results[m]['test_accuracy'] for m in results],
    'AUC-ROC': [results[m]['auc_roc'] for m in results]
})

comparison_df = comparison_df.sort_values('Test Accuracy', ascending=False)
print("Model Comparison:")
comparison_df

In [ ]:
comparison_df_melted = comparison_df.melt(
    id_vars='Model', 
    value_vars=['CV Accuracy', 'Test Accuracy', 'AUC-ROC'],
    var_name='Metric', 
    value_name='Score'
)

plt.figure(figsize=(12, 7))
sns.barplot(x='Model', y='Score', hue='Metric', data=comparison_df_melted)
plt.title('Model Performance Comparison', fontsize=16)
plt.xticks(rotation=45)
plt.ylim(0, 1)
plt.legend(loc='lower right')
plt.tight_layout()
plt.show()

In [ ]:
print("--- Hyperparameter Tuning for Random Forest ---")

param_grid_rf = {
    'n_estimators': [50, 100, 200],
    'max_depth': [None, 10, 20],
    'min_samples_split': [2, 5, 10]
}

grid_rf = GridSearchCV(
    RandomForestClassifier(random_state=42), 
    param_grid_rf, 
    cv=5, 
    scoring='accuracy',
    n_jobs=-1
)

grid_rf.fit(X_train_scaled, y_train)

print(f"Best parameters: {grid_rf.best_params_}")
print(f"Best CV Accuracy: {grid_rf.best_score_:.4f}")

In [ ]:
best_rf = grid_rf.best_estimator_
y_pred_rf = best_rf.predict(X_test_scaled)
y_pred_proba_rf = best_rf.predict_proba(X_test_scaled)[:, 1]

test_acc = accuracy_score(y_test, y_pred_rf)
auc = roc_auc_score(y_test, y_pred_proba_rf)

print(f"Test Accuracy with tuned RF: {test_acc:.4f}")
print(f"AUC-ROC with tuned RF: {auc:.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred_rf))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_rf))

In [ ]:
rf_model = models['Random Forest']
feature_importance = pd.Series(rf_model.feature_importances_, index=X.columns).sort_values(ascending=False)

plt.figure(figsize=(12, 7))
feature_importance.plot(kind='bar', color='steelblue')
plt.title('Random Forest Feature Importance', fontsize=16)
plt.xlabel('Features')
plt.ylabel('Importance Score')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

print("\nTop 10 Important Features (Random Forest):")
print(feature_importance.head(10))

In [ ]:
gb_model = models['Gradient Boosting']
gb_importance = pd.Series(gb_model.feature_importances_, index=X.columns).sort_values(ascending=False)

plt.figure(figsize=(12, 7))
gb_importance.plot(kind='bar', color='darkorange')
plt.title('Gradient Boosting Feature Importance', fontsize=16)
plt.xlabel('Features')
plt.ylabel('Importance Score')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

print("\nTop 10 Important Features (Gradient Boosting):")
print(gb_importance.head(10))

In [ ]:
# Add tuned RF to comparison
comparison_df_updated = comparison_df.copy()

tuned_rf_row = pd.DataFrame({
    'Model': ['Tuned Random Forest'],
    'CV Accuracy': [grid_rf.best_score_],
    'CV Std': [np.std(grid_rf.cv_results_['mean_test_score'])],
    'Test Accuracy': [test_acc],
    'AUC-ROC': [auc]
})

comparison_df_updated = pd.concat([comparison_df_updated, tuned_rf_row], ignore_index=True)
comparison_df_updated = comparison_df_updated.sort_values('Test Accuracy', ascending=False)

print("Updated Model Comparison (with Tuned RF):")
comparison_df_updated

In [ ]:
comparison_updated_melted = comparison_df_updated.melt(
    id_vars='Model', 
    value_vars=['CV Accuracy', 'Test Accuracy', 'AUC-ROC'],
    var_name='Metric', 
    value_name='Score'
)

plt.figure(figsize=(14, 8))
sns.barplot(x='Model', y='Score', hue='Metric', data=comparison_updated_melted)
plt.title('Model Performance Comparison (Including Tuned Random Forest)', fontsize=16)
plt.xticks(rotation=45)
plt.ylim(0, 1)
plt.legend(loc='lower right')
plt.tight_layout()
plt.show()

In [ ]:
print("="*50)
print("FINAL SUMMARY")
print("="*50)

print(f"\nDataset Shape: {df.shape}")
print(f"Total Samples: {len(df)}")
print(f"Alive (0): {df['DEATH_EVENT'].value_counts()[0]}")
print(f"Deceased (1): {df['DEATH_EVENT'].value_counts()[1]}")
print(f"Mortality Rate: {df['DEATH_EVENT'].mean()*100:.2f}%")

print("\n" + "="*50)
print("BEST PERFORMING MODELS:")
print("="*50)

best_accuracy = comparison_df_updated.loc[comparison_df_updated['Test Accuracy'].idxmax()]
best_auc = comparison_df_updated.loc[comparison_df_updated['AUC-ROC'].idxmax()]

print(f"\nHighest Accuracy: {best_accuracy['Model']} - {best_accuracy['Test Accuracy']:.4f}")
print(f"Highest AUC-ROC: {best_auc['Model']} - {best_auc['AUC-ROC']:.4f}")

print("\n" + "="*50)
print("TOP 5 MOST IMPORTANT FEATURES (Random Forest):")
print("="*50)
print(feature_importance.head(5))

print("\n" + "="*50)
print("TOP 5 MOST IMPORTANT FEATURES (Gradient Boosting):")
print("="*50)
print(gb_importance.head(5))

In [ ]:
import joblib

# Save the best performing model (tuned Random Forest)
joblib.dump(best_rf, 'heart_failure_best_model.pkl')
print("Best model saved as 'heart_failure_best_model.pkl'")

# Save the scaler
joblib.dump(scaler, 'scaler.pkl')
print("Scaler saved as 'scaler.pkl'")